# Lora 实战

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

## Step3 数据集预处理

In [4]:
tokenizer = AutoTokenizer.from_pretrained("../../model/langboat/bloom-1b4-zh")
tokenizer

BloomTokenizerFast(name_or_path='../../model/langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [5]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [6]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [7]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [8]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

## Step4 创建模型

In [9]:
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-1b4-zh")

In [10]:
model

BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(46145, 2048)
    (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-23): 24 x BloomBlock(
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
          (dense): Linear(in_features=2048, out_features=2048, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
          (gelu_impl): BloomGelu()
          (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
  )
  (l

In [11]:
for name, parameter in model.named_parameters():
    print(name)

transformer.word_embeddings.weight
transformer.word_embeddings_layernorm.weight
transformer.word_embeddings_layernorm.bias
transformer.h.0.input_layernorm.weight
transformer.h.0.input_layernorm.bias
transformer.h.0.self_attention.query_key_value.weight
transformer.h.0.self_attention.query_key_value.bias
transformer.h.0.self_attention.dense.weight
transformer.h.0.self_attention.dense.bias
transformer.h.0.post_attention_layernorm.weight
transformer.h.0.post_attention_layernorm.bias
transformer.h.0.mlp.dense_h_to_4h.weight
transformer.h.0.mlp.dense_h_to_4h.bias
transformer.h.0.mlp.dense_4h_to_h.weight
transformer.h.0.mlp.dense_4h_to_h.bias
transformer.h.1.input_layernorm.weight
transformer.h.1.input_layernorm.bias
transformer.h.1.self_attention.query_key_value.weight
transformer.h.1.self_attention.query_key_value.bias
transformer.h.1.self_attention.dense.weight
transformer.h.1.self_attention.dense.bias
transformer.h.1.post_attention_layernorm.weight
transformer.h.1.post_attention_layernor

In [12]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 考试有哪些技巧？\n- 有哪些技巧？\n- 准备、应考。\n- 准备什么？\n- 技巧。\n技巧？\n- 技巧。\n- 技巧。\n技巧？\n- 什么技巧？\n什么技巧？\n- 你自己看。\n- 自 己看。\n- 自...\n- 自...\n 自...\n自...\n自...\n自...\n自...\n自...\n自...\n自...\n自...\n自...\n- 那好。\n- 那好。\n- 那好。\n- 那好。\n- 那么好的自 己。\n- 那么好的自 '

## Lora

### PEFT Step1 配置文件

In [13]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(task_type=TaskType.CAUSAL_LM, target_modules=".*\.1.*query_key_value")
config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=8, target_modules='.*\\.1.*query_key_value', lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None)

### PEFT Step2 创建模型

In [14]:
model = get_peft_model(model, config)

In [15]:
config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='../../model/langboat/bloom-1b4-zh', revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=8, target_modules='.*\\.1.*query_key_value', lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None)

In [16]:
for name, parameter in model.named_parameters():
    print(name)

base_model.model.transformer.word_embeddings.weight
base_model.model.transformer.word_embeddings_layernorm.weight
base_model.model.transformer.word_embeddings_layernorm.bias
base_model.model.transformer.h.0.input_layernorm.weight
base_model.model.transformer.h.0.input_layernorm.bias
base_model.model.transformer.h.0.self_attention.query_key_value.weight
base_model.model.transformer.h.0.self_attention.query_key_value.bias
base_model.model.transformer.h.0.self_attention.dense.weight
base_model.model.transformer.h.0.self_attention.dense.bias
base_model.model.transformer.h.0.post_attention_layernorm.weight
base_model.model.transformer.h.0.post_attention_layernorm.bias
base_model.model.transformer.h.0.mlp.dense_h_to_4h.weight
base_model.model.transformer.h.0.mlp.dense_h_to_4h.bias
base_model.model.transformer.h.0.mlp.dense_4h_to_h.weight
base_model.model.transformer.h.0.mlp.dense_4h_to_h.bias
base_model.model.transformer.h.1.input_layernorm.weight
base_model.model.transformer.h.1.input_layer

In [17]:
model.print_trainable_parameters()

trainable params: 720,896 || all params: 1,303,832,576 || trainable%: 0.0553


## Step5 配置训练参数

In [18]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=50,
    num_train_epochs=1
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step6 创建训练器

In [19]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## Step7 模型训练

In [20]:
trainer.train()

  0%|          | 0/3357 [00:00<?, ?it/s]

{'loss': 3.0116, 'grad_norm': 0.9263869524002075, 'learning_rate': 4.925528745904081e-05, 'epoch': 0.01}
{'loss': 2.6755, 'grad_norm': 0.6229520440101624, 'learning_rate': 4.851057491808162e-05, 'epoch': 0.03}
{'loss': 2.606, 'grad_norm': 3.407592296600342, 'learning_rate': 4.776586237712243e-05, 'epoch': 0.04}
{'loss': 2.4849, 'grad_norm': 1.1227200031280518, 'learning_rate': 4.702114983616324e-05, 'epoch': 0.06}
{'loss': 2.443, 'grad_norm': 1.6021744012832642, 'learning_rate': 4.627643729520406e-05, 'epoch': 0.07}
{'loss': 2.4749, 'grad_norm': 0.9554081559181213, 'learning_rate': 4.553172475424487e-05, 'epoch': 0.09}
{'loss': 2.4314, 'grad_norm': 1.1485419273376465, 'learning_rate': 4.478701221328567e-05, 'epoch': 0.1}
{'loss': 2.4541, 'grad_norm': 3.7391140460968018, 'learning_rate': 4.404229967232648e-05, 'epoch': 0.12}
{'loss': 2.3961, 'grad_norm': 1.2225232124328613, 'learning_rate': 4.32975871313673e-05, 'epoch': 0.13}
{'loss': 2.4417, 'grad_norm': 0.7741189002990723, 'learning_

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.3952, 'grad_norm': 1.408542275428772, 'learning_rate': 4.180816204944891e-05, 'epoch': 0.16}
{'loss': 2.3604, 'grad_norm': 2.1616690158843994, 'learning_rate': 4.1063449508489726e-05, 'epoch': 0.18}
{'loss': 2.3857, 'grad_norm': 1.2304316759109497, 'learning_rate': 4.0318736967530536e-05, 'epoch': 0.19}
{'loss': 2.3355, 'grad_norm': 1.2460819482803345, 'learning_rate': 3.9574024426571345e-05, 'epoch': 0.21}
{'loss': 2.3879, 'grad_norm': 0.9961667656898499, 'learning_rate': 3.8829311885612155e-05, 'epoch': 0.22}
{'loss': 2.3755, 'grad_norm': 1.4235553741455078, 'learning_rate': 3.8084599344652965e-05, 'epoch': 0.24}
{'loss': 2.3172, 'grad_norm': 3.1362812519073486, 'learning_rate': 3.7339886803693774e-05, 'epoch': 0.25}
{'loss': 2.4095, 'grad_norm': 1.7865102291107178, 'learning_rate': 3.659517426273459e-05, 'epoch': 0.27}
{'loss': 2.308, 'grad_norm': 1.5255850553512573, 'learning_rate': 3.5850461721775394e-05, 'epoch': 0.28}
{'loss': 2.2799, 'grad_norm': 3.7476320266723633, 

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


...

{'loss': 2.251, 'grad_norm': 2.1522679328918457, 'learning_rate': 1.2019660411081324e-05, 'epoch': 0.76}
{'loss': 2.3021, 'grad_norm': 3.2234232425689697, 'learning_rate': 1.1274947870122133e-05, 'epoch': 0.77}
{'loss': 2.2774, 'grad_norm': 2.2030670642852783, 'learning_rate': 1.0530235329162943e-05, 'epoch': 0.79}
{'loss': 2.2054, 'grad_norm': 4.205674171447754, 'learning_rate': 9.785522788203753e-06, 'epoch': 0.8}
{'loss': 2.2177, 'grad_norm': 4.143559455871582, 'learning_rate': 9.040810247244564e-06, 'epoch': 0.82}
{'loss': 2.2825, 'grad_norm': 1.7104378938674927, 'learning_rate': 8.296097706285374e-06, 'epoch': 0.83}
{'loss': 2.2162, 'grad_norm': 2.3052055835723877, 'learning_rate': 7.5513851653261844e-06, 'epoch': 0.85}
{'loss': 2.2409, 'grad_norm': 2.514340877532959, 'learning_rate': 6.806672624366994e-06, 'epoch': 0.86}
{'loss': 2.2597, 'grad_norm': 1.8104277849197388, 'learning_rate': 6.061960083407805e-06, 'epoch': 0.88}
{'loss': 2.2656, 'grad_norm': 2.4474027156829834, 'learn

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.2492, 'grad_norm': 2.1903622150421143, 'learning_rate': 4.572535001489425e-06, 'epoch': 0.91}
{'loss': 2.2898, 'grad_norm': 1.9683070182800293, 'learning_rate': 3.827822460530236e-06, 'epoch': 0.92}
{'loss': 2.2226, 'grad_norm': 2.6794893741607666, 'learning_rate': 3.0831099195710457e-06, 'epoch': 0.94}
{'loss': 2.1967, 'grad_norm': 4.845311641693115, 'learning_rate': 2.338397378611856e-06, 'epoch': 0.95}
{'loss': 2.2883, 'grad_norm': 1.4882490634918213, 'learning_rate': 1.593684837652666e-06, 'epoch': 0.97}
{'loss': 2.232, 'grad_norm': 1.6083259582519531, 'learning_rate': 8.489722966934764e-07, 'epoch': 0.98}
{'loss': 2.1947, 'grad_norm': 3.476099967956543, 'learning_rate': 1.0425975573428657e-07, 'epoch': 1.0}
{'train_runtime': 2860.7888, 'train_samples_per_second': 9.388, 'train_steps_per_second': 1.173, 'train_loss': 2.317561272606098, 'epoch': 1.0}


/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=3357, training_loss=2.317561272606098, metrics={'train_runtime': 2860.7888, 'train_samples_per_second': 9.388, 'train_steps_per_second': 1.173, 'total_flos': 1.4772921222119424e+16, 'train_loss': 2.317561272606098, 'epoch': 0.9999255342914588})

## Step8 模型推理

In [22]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 考试技巧是指在考试的过程中如何提高试卷的命中率和通过率。具体来说，以下技巧可以帮助你：\n\n1. 了解考试和你的知识点：\n\n在做题的时候，你需要知道你所面临的是何种类型题目，并且找到那些与该类型题目的基本概念相关的知识点。例如，在数学考试中可能需要记住加法和减法。因此，在你做题前准备好你所需要使用的基本概念，并尝试在考试中回答带有该词汇的问题。\n\n2. 做练习和模拟测试：\n\n在你'